# GAIL — Generative Adversarial Imitation Learning (Google Colab)

Trains a PPO generator via **GAIL** ([docs](https://imitation.readthedocs.io/en/latest/algorithms/gail.html)) on human demonstrations for the `OpenElectricKettleLid` task.

### What GAIL does differently from plain BC
| | BC (MSE) | GAIL |
|---|---|---|
| Training signal | Supervised MSE on actions | Discriminator reward (adversarial) |
| Env interaction | None (offline) | Yes — PPO rolls out in the env |
| Distribution shift | Prone to compounding errors | More robust |
| Cost | Very cheap | Moderate (needs RL loop) |

### Steps
1. GPU check
2. Clone repo → branch `b313`
3. Install dependencies
4. Download dataset (`~200 MB`)
5. Configure paths
6. Train GAIL
7. Results + plots
8. Save to Google Drive *(optional)*

> **Runtime:** `Runtime → Change runtime type → T4 GPU` before starting.

---
## 1. GPU Check

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if r.returncode == 0:
    print(r.stdout)
else:
    print('⚠️  No GPU detected — training will use CPU (much slower).')

---
## 2. Clone Repository

In [ ]:
# Fill in a GitHub PAT only if the repo is private.
GITHUB_TOKEN = ''   # e.g. 'ghp_xxxxxxxxxxxxxxxxxxxx'

REPO    = 'sergio-contente/RoboCasa-Project'
BRANCH  = 'b313'
WORKDIR = '/content/RoboCasa-Project'

import os, subprocess

if GITHUB_TOKEN:
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'
else:
    clone_url = f'https://github.com/{REPO}.git'

if not os.path.isdir(WORKDIR):
    subprocess.run(
        ['git', 'clone', '--recurse-submodules', '-b', BRANCH, clone_url, WORKDIR],
        check=True
    )
else:
    print(f'{WORKDIR} already exists — pulling.')
    subprocess.run(['git', '-C', WORKDIR, 'pull'], check=True)
    subprocess.run(
        ['git', '-C', WORKDIR, 'submodule', 'update', '--init', '--recursive'],
        check=True
    )

result = subprocess.run(
    ['git', '-C', WORKDIR, 'log', '--oneline', '-3'],
    capture_output=True, text=True
)
print(result.stdout)

---
## 3. Install Dependencies

> **After this cell finishes, the kernel must be restarted.** A helper cell at the end of this section does it automatically.

In [ ]:
import os, subprocess
WORKDIR = '/content/RoboCasa-Project'

# robosuite must be installed before robocasa (robocasa depends on it)
print('Installing robosuite...')
os.system(f'pip install -e {WORKDIR}/deps/robosuite -q')

print('Installing robocasa...')
os.system(f'pip install -e {WORKDIR}/deps/robocasa -q')

# imitation==1.0.1 requires stable-baselines3==2.2.1
print('Installing imitation + compatible stable-baselines3...')
os.system('pip install "imitation==1.0.1" "stable-baselines3==2.2.1" -q')

print('Installing remaining utilities...')
os.system('pip install pyarrow -q')

# Set up robocasa macros (we override DATASET_BASE_PATH in Section 4)
print('Setting up robocasa macros...')
subprocess.run(
    ['python', '-m', 'robocasa.scripts.setup_macros'],
    input='y\n', text=True, cwd=WORKDIR,
)

# Download kitchen visual assets (textures, meshes) required by the simulator
print('Downloading kitchen assets...')
subprocess.run(
    ['python', '-m', 'robocasa.scripts.download_kitchen_assets'],
    input='y\n', text=True, cwd=WORKDIR,
)

print('\nAll done. Restarting kernel...')

In [ ]:
# ⚠️  Restart kernel so fresh installs are active.
# Continue from Section 4 after the restart.
import os
os.kill(os.getpid(), 9)

---
## 4. Download Dataset

Downloads the `pretrain / human` split (~200 MB) using the official RoboCasa download script.

In [ ]:
import os, sys, subprocess
WORKDIR      = '/content/RoboCasa-Project'
DATASET_BASE = '/content/datasets'
os.makedirs(DATASET_BASE, exist_ok=True)

# Point RoboCasa to our dataset folder (must happen before the first robocasa import)
import robocasa
macros_private = os.path.join(robocasa.__path__[0], 'macros_private.py')
with open(macros_private, 'w') as f:
    f.write(f'DATASET_BASE_PATH = "{DATASET_BASE}"\n')
print(f'DATASET_BASE_PATH = {DATASET_BASE}')

DATASET_PATH = (
    f'{DATASET_BASE}/v1.0/pretrain/atomic/OpenElectricKettleLid/20250820/lerobot'
)

if os.path.isdir(os.path.join(DATASET_PATH, 'data')):
    n = len(list(__import__('pathlib').Path(DATASET_PATH).glob('data/*/episode_*.parquet')))
    print(f'Dataset already present — {n} episodes found.')
else:
    print('Downloading dataset (pretrain / human)...')
    result = subprocess.run(
        [
            'python', '-m', 'robocasa.scripts.download_datasets',
            '--tasks',  'OpenElectricKettleLid',
            '--split',  'pretrain',
            '--source', 'human',
        ],
        input='y\n',         # answer the interactive "Proceed? (y/n)" prompt
        text=True,
        capture_output=False,
        env={**os.environ, 'ROBOCASA_DATASET_BASE_PATH': DATASET_BASE},
    )
    if result.returncode != 0:
        raise RuntimeError('Dataset download failed — check output above.')
    print('Download complete.')

In [ ]:
# Verify dataset structure
from pathlib import Path
ds = Path(DATASET_PATH)
print('Dataset structure:')
for d in ['data', 'extras', 'meta']:
    status = '✓' if (ds / d).is_dir() else '✗ missing'
    print(f'  {d}/  {status}')

episodes = list(ds.glob('data/*/episode_*.parquet'))
print(f'\nEpisodes: {len(episodes)}')
print(f'modality.json: {"✓" if (ds / "meta" / "modality.json").exists() else "✗ (bundled fallback will be used)"}')

---
## 5. Configure Project Paths

In [ ]:
import sys, os
WORKDIR      = '/content/RoboCasa-Project'
DATASET_BASE = '/content/datasets'
OUTPUT_DIR   = f'{WORKDIR}/results'

os.chdir(WORKDIR)
if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Working dir  : {os.getcwd()}')
print(f'Dataset base : {DATASET_BASE}')
print(f'Results dir  : {OUTPUT_DIR}')

import torch
print(f'Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')

---
## 6. Hyperparameters

In [ ]:
# ── Edit before training ──────────────────────────────────────────────────
N_DEMO_EPISODES  = 50       # expert episodes to load
TOTAL_TIMESTEPS  = 500_000  # total GAIL training timesteps
N_ENVS           = 4        # parallel environments for PPO rollouts
LEARNING_RATE    = 3e-4
N_STEPS          = 2048     # PPO steps per env per update
BATCH_SIZE       = 256      # PPO mini-batch size
DEMO_BATCH       = 1024     # expert transitions per discriminator update
N_DISC_UPDATES   = 4        # discriminator gradient steps per round
N_EVAL_EPISODES  = 20       # rollout episodes after training
# ─────────────────────────────────────────────────────────────────────────

---
## 7. Train GAIL

Each *round* alternates between:
1. **Generator step** — PPO collects `N_STEPS × N_ENVS` transitions; reward from discriminator
2. **Discriminator step** — `N_DISC_UPDATES` gradient steps classifying expert vs learner

In [ ]:
import train_gail
import importlib; importlib.reload(train_gail)

train_gail.main([
    '--episodes',       str(N_DEMO_EPISODES),
    '--timesteps',      str(TOTAL_TIMESTEPS),
    '--n-envs',         str(N_ENVS),
    '--lr',             str(LEARNING_RATE),
    '--n-steps',        str(N_STEPS),
    '--batch-size',     str(BATCH_SIZE),
    '--demo-batch',     str(DEMO_BATCH),
    '--n-disc-updates', str(N_DISC_UPDATES),
    '--eval-ep',        str(N_EVAL_EPISODES),
    '--dataset-root',   DATASET_BASE,
    '--output',         OUTPUT_DIR,
])

---
## 8. Results

In [ ]:
import json
from pathlib import Path

eval_file = Path(OUTPUT_DIR) / 'gail_eval.json'
if not eval_file.exists():
    print('gail_eval.json not found — run Section 7 first.')
else:
    metrics = json.loads(eval_file.read_text())
    print('GAIL Evaluation Results (true environment reward)')
    print('─' * 40)
    for k, v in metrics.items():
        if isinstance(v, float):
            print(f'  {k:<28}: {v:.4f}')
        else:
            print(f'  {k:<28}: {v}')

In [ ]:
# Visualise discriminator log (reward proxy over time)
import matplotlib.pyplot as plt
import os

log_dir = os.path.join(OUTPUT_DIR, 'gail_logs')
monitor_files = list(Path(log_dir).glob('**/*.monitor.csv')) if os.path.isdir(log_dir) else []

if monitor_files:
    import pandas as pd
    df = pd.concat([pd.read_csv(f, skiprows=1) for f in monitor_files])
    df = df.sort_values('t')

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(df['t'], df['r'].rolling(50, min_periods=1).mean(),
            color='#4C72B0', linewidth=1.5, label='Disc. reward (rolling 50)')
    ax.set_xlabel('Wall-clock time (s)')
    ax.set_ylabel('Episode reward (discriminator)')
    ax.set_title('GAIL — Generator Discriminator Reward', fontweight='bold')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/gail_disc_reward.png', dpi=150)
    plt.show()
else:
    print('No monitor logs found — reward curve unavailable.')

---
## 9. Save to Google Drive *(optional)*

In [ ]:
SAVE_TO_DRIVE = False          # ← set True to enable
DRIVE_FOLDER  = 'MyDrive/RoboCasa'

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    import shutil
    dest = f'/content/drive/{DRIVE_FOLDER}'
    os.makedirs(dest, exist_ok=True)

    for fname in ['gail_model.zip', 'gail_eval.json', 'gail_disc_reward.png']:
        src = Path(OUTPUT_DIR) / fname
        if src.exists():
            shutil.copy(src, dest)
            print(f'Copied {fname} → {dest}')
else:
    print('Drive save skipped (set SAVE_TO_DRIVE = True to enable).')

---
## Summary

| File | Description |
|------|-------------|
| `results/gail_model.zip` | Trained PPO generator — load with `PPO.load(path)` |
| `results/gail_eval.json` | Final evaluation metrics (true env reward) |
| `results/gail_logs/` | Training logs (discriminator, generator losses) |

### How to load the model later
```python
from stable_baselines3 import PPO
model = PPO.load('results/gail_model')
action, _ = model.predict(obs, deterministic=True)
```

### Next steps
Fine-tune the GAIL policy with dense reward or curriculum learning using `train_il_rl_rs.py` — pass `--bc-model` with the generator weights extracted from `gail_model.zip`.